In [1]:
import os
from dotenv import load_dotenv
import memengine

load_dotenv()
print("memengine:", memengine)
print("memengine exports:", [n for n in ("FUMemory","STMemory","LTMemory","MBMemory","MemoryConfig") if hasattr(memengine,n)])

from memengine import MemoryConfig, FUMemory, STMemory, LTMemory, MBMemory

d:\Anaconda\envs\py312pt291cu128\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


memengine: <module 'memengine' from 'd:\\Anaconda\\envs\\py312pt291cu128\\Lib\\site-packages\\memengine\\__init__.py'>
memengine exports: ['FUMemory', 'STMemory', 'LTMemory', 'MBMemory', 'MemoryConfig']


In [2]:
import json
from pathlib import Path

DATA_PATH = Path("locomo10.json")
assert DATA_PATH.exists(), f"Not found: {DATA_PATH.resolve()}"

raw = json.loads(DATA_PATH.read_text(encoding="utf-8"))
print("items:", len(raw))
print("keys of first item:", list(raw[0].keys()))

qa = raw[0]["qa"]
print("qa count:", len(qa))
print("sample qa[0] keys:", list(qa[0].keys()))

# Turn qa pairs into memory observations (plain text)
# (Some entries may miss the 'answer' field; we skip those.)
qa_valid = [x for x in qa if "answer" in x]
observations = [
    f"Q: {x['question']}\nA: {x['answer']}\nEvidence: {', '.join(x.get('evidence', []))}"
    for x in qa_valid
]

# Pick one query to test recall (use an existing question)
query = qa_valid[0]["question"]
print("query:", query)
print("observation[0]:\n", observations[0])

items: 10
keys of first item: ['qa', 'conversation', 'event_summary', 'observation', 'session_summary', 'sample_id']
qa count: 199
sample qa[0] keys: ['question', 'answer', 'evidence', 'category']
query: When did Caroline go to the LGBTQ support group?
observation[0]:
 Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3


In [3]:
def make_common_config(*, usable_gpu: str = "", display_method: str = "ScreenDisplay") -> dict:
    # Minimal, runnable configs derived from the official open-source defaults,
    # but adapted to run locally without external model paths.
    return {
        "global_config": {"usable_gpu": usable_gpu},
        "storage": {},
        "display": {
            "method": display_method,
            "prefix": "----- Current Memory Start (%s) -----",
            "suffix": "----- Current Memory End -----",
            "key_format": "(%s)",
            "key_value_sep": "\n",
            "item_sep": "\n",
            # FileDisplay-only arg (ignored by ScreenDisplay)
            "output_path": "logs/sample.log",
        },
        "recall": {
            "truncation": {
                "method": "LMTruncation",
                "mode": "word",
                "number": 256,
                "path": "",  # only used for token-based truncation
            },
            "utilization": {
                "method": "ConcateUtilization",
                "prefix": "[Memory Start]",
                "suffix": "[Memory End]",
                "list_config": {"index": True, "sep": "\n"},
                "dict_config": {"key_format": "(%s)", "key_value_sep": "\n", "item_sep": "\n"},
            },
            "empty_memory": "None",
        },
        "store": {},
    }


def make_text_retrieval_config(*, topk: int = 5, st_model: str = "sentence-transformers/all-MiniLM-L6-v2") -> dict:
    # LTMemory / MBMemory rely on embeddings; this uses SentenceTransformers.
    return {
        "method": "TextRetrieval",
        "encoder": {
            "method": "STEncoder",
            "name": st_model,
            "dimension": 384,
            "path": st_model,
        },
        "mode": "cosine",
        "topk": topk,
    }


def run_memory(
    memory,
    obs_list,
    query_text,
    *,
    with_time: bool = False,
    fixed_time: int | None = None,
    time_bucket: int = 1,
):
    memory.reset()
    for i, obs in enumerate(obs_list):
        if with_time:
            if fixed_time is not None:
                t = fixed_time
            else:
                tb = max(1, int(time_bucket))
                t = i // tb
            memory.store({"text": obs, "time": t})
        else:
            memory.store(obs)
    return memory.recall(query_text)

In [4]:
# --- FUMemory (Full / long-context) ---
fu_cfg = make_common_config()
fu_cfg["name"] = "FUMemory"
fu_cfg["store"] = {"method": "FUMemoryStore"}
fu_cfg["recall"]["method"] = "FUMemoryRecall"

fu = FUMemory(MemoryConfig(fu_cfg))
fu_ans = run_memory(fu, observations[:30], query)
print("FUMemory recall result:\n", fu_ans)

# Optional: visualize internal storage
fu.display()

FUMemory recall result:
 [Memory Start]
[0] Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
[1] Q: When did Melanie paint a sunrise?
A: 2022
Evidence: D1:12
[2] Q: What fields would Caroline be likely to pursue in her educaton?
A: Psychology, counseling certification
Evidence: D1:9, D1:11
[3] Q: What did Caroline research?
A: Adoption agencies
Evidence: D2:8
[4] Q: What is Caroline's identity?
A: Transgender woman
Evidence: D1:5
[5] Q: When did Melanie run a charity race?
A: The sunday before 25 May 2023
Evidence: D2:1
[6] Q: When is Melanie planning on going camping?
A: June 2023
Evidence: D2:7
[7] Q: What is Caroline's relationship status?
A: Single
Evidence: D3:13, D2:14
[8] Q: When did Caroline give a speech at a school?
A: The week before 9 June 2023
Evidence: D3:1
[9] Q: When did Caroline meet up with her friends, family, and mentors?
A: The week before 9 June 2023
Evidence: D3:11
[10] Q: How long has Caroline had her current group of friends for?

In [5]:
# --- STMemory (Short-term / recent window) ---
st_cfg = make_common_config()
st_cfg["name"] = "STMemory"
st_cfg["store"] = {"method": "LTMemoryStore"}
st_cfg["recall"].update({
    "method": "STMemoryRecall",
    "time_retrieval": {"method": "TimeRetrieval", "mode": "raw", "topk": 5},
})

st = STMemory(MemoryConfig(st_cfg))
st_ans = run_memory(st, observations[:30], query)
print("STMemory recall result:\n", st_ans)

st.display()

STMemory recall result:
 [Memory Start]
[0] Q: When did Melanie go to the pottery workshop?
A: The Friday before 15 July 2023
Evidence: D8:2
[1] Q: When did Caroline go to the adoption meeting?
A: The friday before 15 July 2023
Evidence: D8:9
[2] Q: Would Caroline pursue writing as a career option?
A: LIkely no; though she likes reading, she wants to be a counselor
Evidence: D7:5, D7:9
[3] Q: When did Melanie read the book "nothing is impossible"?
A: 2022
Evidence: D7:8
[4] Q: When did Caroline go to the LGBTQ conference?
A: 10 July 2023
Evidence: D7:1
[Memory End]
----- Current Memory Start (30) -----
(Memory Storage)
[Memory Entity 0]
text: Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
counter_id: 0
[Memory Entity 1]
text: Q: When did Melanie paint a sunrise?
A: 2022
Evidence: D1:12
counter_id: 1
[Memory Entity 2]
text: Q: What fields would Caroline be likely to pursue in her educaton?
A: Psychology, counseling certification
Evidence: D1:9, D1:11
co

In [6]:
# --- LTMemory (Long-term / embedding retrieval) ---
lt_cfg = make_common_config()
lt_cfg["name"] = "LTMemory"
lt_cfg["store"] = {"method": "LTMemoryStore"}
lt_cfg["recall"].update({
    "method": "LTMemoryRecall",
    "text_retrieval": make_text_retrieval_config(topk=5),
})

lt = LTMemory(MemoryConfig(lt_cfg))
lt_ans = run_memory(lt, observations[:80], query)
print("LTMemory recall result:\n", lt_ans)

lt.display()

LTMemory recall result:
 [Memory Start]
[0] Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
[1] Q: When did Caroline go to the LGBTQ conference?
A: 10 July 2023
Evidence: D7:1
[2] Q: What LGBTQ+ events has Caroline participated in?
A: Pride parade, school speech, support group
Evidence: D5:1, D8:17, D3:1, D1:3
[3] Q: In what ways is Caroline participating in the LGBTQ community?
A: Joining activist group, going to pride parades, participating in an art show, mentoring program
Evidence: D10:3, D5:1, D9:12, D9:2
[4] Q: What career path has Caroline decided to persue?
A: counseling or mental health for Transgender people
Evidence: D4:13, D1:11
[Memory End]
----- Current Memory Start (80) -----
(Memory Storage)
[Memory Entity 0]
text: Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
counter_id: 0
[Memory Entity 1]
text: Q: When did Melanie paint a sunrise?
A: 2022
Evidence: D1:12
counter_id: 1
[Memory Entity 2]
text: Q: What 

In [7]:
# --- MBMemory (MemoryBank) ---
# MBMemory will summarize when the `time` bucket changes.
# In this notebook we DO want to trigger summarization, but we avoid summarizing on every item
# by bucketing times (e.g., one summary per 20 observations).

mb_cfg = make_common_config()
mb_cfg["name"] = "MBMemory"
mb_cfg["store"] = {
    "method": "MBMemoryStore",
    "summarizer": {
        "method": "LLMSummarizer",
        "LLM_config": {
            "method": "APILLM",
            "name": "arcee-ai/trinity-large-preview:free",
            "api_key": os.environ["OPENROUTER_API_KEY"],
            "base_url": "https://openrouter.ai/api/v1",
            "temperature": 0.0,
        },
        "prompt": {
            "template": "Content: {content}\nSummarize the above content concisely, extracting the main themes and key information.",
            "input_variables": ["content"],
        },
    },
}
mb_cfg["recall"].update({
    "method": "MBMemoryRecall",
    "text_retrieval": make_text_retrieval_config(topk=5),
    # omit "forget" in this basic demo to make results deterministic/visible
})

mb = MBMemory(MemoryConfig(mb_cfg))

# OpenRouter/free-tier models can occasionally return empty/None content.
# MBMemory's pipeline assumes summarizer always returns a non-empty string;
# if it returns None, embedding will crash.
#
# Important: MBMemoryStore.reset() expects `summarizer` to have a `.reset()` method,
# so we wrap it in a small callable object rather than replacing it with a bare function.

class SafeSummarizer:
    def __init__(self, inner):
        self.inner = inner

    def reset(self):
        r = getattr(self.inner, "reset", None)
        if callable(r):
            r()

    def __call__(self, content):
        for _ in range(2):
            try:
                s = self.inner(content)
            except Exception:
                s = None
            if isinstance(s, str) and s.strip():
                return s

        text = content if isinstance(content, str) else str(content)
        text = text.strip()
        if not text:
            return "Summary (fallback): <empty content>"
        return "Summary (fallback): " + (text[:600] + ("..." if len(text) > 600 else ""))

mb.store_op.summarizer = SafeSummarizer(mb.store_op.summarizer)

mb_ans = run_memory(mb, observations[:80], query, with_time=True, time_bucket=20)
print("MBMemory recall result:\n", mb_ans)

mb.display()

MBMemory recall result:
 [Memory Start]
[0] None
[1] Q: When did Caroline go to the LGBTQ support group?
A: 7 May 2023
Evidence: D1:3
[2] Q: When did Caroline go to the LGBTQ conference?
A: 10 July 2023
Evidence: D7:1
[3] Q: What LGBTQ+ events has Caroline participated in?
A: Pride parade, school speech, support group
Evidence: D5:1, D8:17, D3:1, D1:3
[4] Q: In what ways is Caroline participating in the LGBTQ community?
A: Joining activist group, going to pride parades, participating in an art show, mentoring program
Evidence: D10:3, D5:1, D9:12, D9:2
[5] Q: What career path has Caroline decided to persue?
A: counseling or mental health for Transgender people
Evidence: D4:13, D1:11
[6] Q: What transgender-specific events has Caroline attended?
A: Poetry reading, conference
Evidence: D17:19, D15:13
[7] Q: When did Caroline join a new activist group?
A: The Tuesday before 20 July 2023
Evidence: D10:3
[8] Q: When is Caroline going to the transgender conference?
A: July 2023
Evidence: D5:1